# Cirq (Google) — Framework de Computação Quântica

## O que é o Cirq?

**Cirq** é o framework open-source de computação quântica do Google.
É projetado para trabalhar em **nível mais baixo** que o Qiskit,
dando controle granular sobre *como* e *quando* cada porta é executada.

## Filosofia: Moments (Camadas Temporais)

A diferença central do Cirq é o conceito de **`cirq.Moment`**:

```
Momento 0    Momento 1    Momento 2
────────────────────────────────────
[H(q0)]      [CNOT(q0,q1)] [M(q0,q1)]
```

Enquanto o Qiskit infere o paralelismo automaticamente,
**no Cirq você define explicitamente**: *esta porta acontece neste momento*.

| Conceito Cirq | Equivalente Qiskit |
|---------------|--------------------|
| `cirq.LineQubit` | Índice de qubit |
| `cirq.Moment` | Camada/Barreira |
| `cirq.Circuit` | `QuantumCircuit` |
| `cirq.Simulator()` | `AerSimulator()` |

## Quando usar Cirq?

```
✅ Hardware Google Quantum AI (Sycamore, Willow)
✅ Pesquisa de compilação de baixo nível
✅ Controle preciso do scheduling de portas
✅ Algoritmos NISQ (Noisy Intermediate-Scale Quantum)
✅ Física experimental — sabe o mapa de qubits físicos

❌ Projetos gerais com ecossistema amplo (use Qiskit)
❌ QML com autodiferenciação (use PennyLane)
❌ Hardware IBM
```

## Exemplo 1 — Bell State com Moments explícitos

Vamos criar o mesmo Bell State do Qiskit, mas agora **explicitando os Moments** —
o diferencial filosófico do Cirq.

### Tipos de qubits no Cirq
- `cirq.LineQubit(i)`: qubit na posição i de uma linha (1D)
- `cirq.GridQubit(row, col)`: qubit na grade 2D (como no hardware Google)
- `cirq.NamedQubit('nome')`: qubit com nome simbólico

In [1]:
import cirq

# Definição dos qubits
q0, q1 = cirq.LineQubit.range(2)

# --- Opção 1: Cirq infere os Moments automaticamente ---
circuit_auto = cirq.Circuit([
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key='resultado'),
])

print('=== Circuit (inferência automática) ===')
print(circuit_auto)
print()

# --- Opção 2: Defina os Moments manualmente ---
circuit_manual = cirq.Circuit([
    cirq.Moment([cirq.H(q0)]),           # Momento 0: Hadamard em q0
    cirq.Moment([cirq.CNOT(q0, q1)]),    # Momento 1: CNOT
    cirq.Moment([cirq.measure(q0, q1, key='resultado')]),  # Momento 2: medição
])

print('=== Circuit (Moments explícitos) ===')
print(circuit_manual)

=== Circuit (inferência automática) ===
0: ───H───@───M('resultado')───
          │   │
1: ───────X───M────────────────

=== Circuit (Moments explícitos) ===
0: ───H───@───M('resultado')───
          │   │
1: ───────X───M────────────────


## Exemplo 2 — Simulação com cirq.Simulator

O `cirq.Simulator` permite dois modos:
- **`simulate()`**: retorna o **vetor de estado** (statevector) **sem** medição
- **`run()`**: executa o circuito N vezes (shots) e retorna as **medições**

In [2]:
import cirq
import numpy as np

q0, q1 = cirq.LineQubit.range(2)
sim = cirq.Simulator()

# ---- Circuito Bell (sem medição para obter statevector) ----
bell = cirq.Circuit([cirq.H(q0), cirq.CNOT(q0, q1)])

# simulate() → retorna vetor de estado exato
resultado_sv = sim.simulate(bell)
print('Vetor de estado (statevector):')
print(resultado_sv.final_state_vector)
# Esperado: [1/√2, 0, 0, 1/√2] ≈ [0.707, 0, 0, 0.707]

# ---- Circuito com medição ----
bell_com_medicao = cirq.Circuit([
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key='bits'),
])

# run() → executa N vezes e conta resultados
resultado_run = sim.run(bell_com_medicao, repetitions=1024)
print('\nContagens de medição:')
print(resultado_run.histogram(key='bits'))
# 0 = '00', 3 = '11'

Vetor de estado (statevector):
[0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]

Contagens de medição:
Counter({0: 516, 3: 508})


## Exemplo 3 — GridQubit: Mapa de Hardware Real do Google

O hardware Google usa uma **grade 2D** de qubits. Cada qubit tem
coordenadas `(row, col)`. A conectividade é limitada a vizinhos adjacentes.

Usar `GridQubit` significa que você está pensando como o **compilador**
que vai rodar no chip físico — esse é o nível de controle que o Cirq oferece.

### Topologia do processador Google Sycamore (53 qubits)
```
  . Q Q . .
. Q Q Q Q .
Q Q Q Q Q Q
. Q Q Q Q .
  . Q Q . .
```
Cada Q pode operar com seus 4 vizinhos ortogonais.

In [3]:
import cirq

# GridQubits simulam qubits físicos em grade 2D
q00 = cirq.GridQubit(0, 0)
q01 = cirq.GridQubit(0, 1)
q10 = cirq.GridQubit(1, 0)
q11 = cirq.GridQubit(1, 1)

# Circuito que cria 2 pares de Bell em paralelo no mesmo Momento
# Par 1: q00 e q01 | Par 2: q10 e q11
circuit = cirq.Circuit([
    # Momento 0: Hadamard em paralelo nos 4 qubits
    cirq.Moment([cirq.H(q00), cirq.H(q10)]),

    # Momento 1: CNOT em paralelo (par 1 e par 2 ao mesmo tempo)
    cirq.Moment([cirq.CNOT(q00, q01), cirq.CNOT(q10, q11)]),

    # Momento 2: Medição de todos
    cirq.Moment([cirq.measure(q00, q01, key='par1'),
                 cirq.measure(q10, q11, key='par2')]),
])

print(circuit)

sim = cirq.Simulator()
resultado = sim.run(circuit, repetitions=512)
print('\nPar 1:', resultado.histogram(key='par1'))
print('Par 2:', resultado.histogram(key='par2'))
# Ambos devem mostrar apenas 0 ('00') e 3 ('11')

(0, 0): ───H───@───M('par1')───
               │   │
(0, 1): ───────X───M───────────

(1, 0): ───H───@───M('par2')───
               │   │
(1, 1): ───────X───M───────────

Par 1: Counter({0: 268, 3: 244})
Par 2: Counter({3: 258, 0: 254})


## Exemplo 4 — Portas Nativas do Google (FSim, PhasedXZ)

O hardware Google tem um **gate set nativo** diferente do IBM:

| Porta | Descrição |
|-------|-----------|
| `FSim(θ, φ)` | Fermionic Simulation — porta de 2 qubits mais geral do Google |
| `PhasedXZGate` | Rotação + fase: gate de 1 qubit mais eficiente |
| `CZ` | Controlled-Z nativo no Sycamore |
| `ISWAP` | iSWAP — gate de 2 qubits nativo |

O `FSim(θ=π/2, φ=π/6)` é literalmente o gate físico do chip Sycamore —
resultado de acoplamento superconductor real.

In [4]:
import cirq
import numpy as np

q0, q1 = cirq.LineQubit.range(2)

# --- FSim: gate nativo do processador Google ---
# theta: ângulo de swap | phi: ângulo de fase condicional
fsim = cirq.FSimGate(theta=np.pi/2, phi=np.pi/6)
print('Matriz unitária do FSim(π/2, π/6):')
print(np.round(cirq.unitary(fsim), 3))

# --- CZ Gate: Controlled-Z nativo no Sycamore ---
# Mais eficiente que CNOT no hardware Google
circuit_cz = cirq.Circuit([
    cirq.H(q0),
    cirq.H(q1),
    cirq.CZ(q0, q1),           # CZ nativo
    cirq.H(q1),                 # CZ + H = CNOT logicamente
    cirq.measure(q0, q1, key='medicao'),
])

print('\nCircuito com CZ (equivalente a CNOT):')
print(circuit_cz)

# --- iSWAP: troca qubits com fase ---
circuit_iswap = cirq.Circuit([
    cirq.X(q0),                 # prepara q0 = |1⟩
    cirq.ISWAP(q0, q1),        # troca q0 e q1 com fase i
    cirq.measure(q0, q1, key='iswap'),
])
print('\nCircuito iSWAP (|10⟩ → i|01⟩):')
print(circuit_iswap)

sim = cirq.Simulator()
print('\nResultado iSWAP:')
print(sim.run(circuit_iswap, repetitions=10))

Matriz unitária do FSim(π/2, π/6):
[[1.   +0.j  0.   +0.j  0.   +0.j  0.   +0.j ]
 [0.   +0.j  0.   +0.j  0.   -1.j  0.   +0.j ]
 [0.   +0.j  0.   -1.j  0.   +0.j  0.   +0.j ]
 [0.   +0.j  0.   +0.j  0.   +0.j  0.866-0.5j]]

Circuito com CZ (equivalente a CNOT):
0: ───H───@───────M('medicao')───
          │       │
1: ───H───@───H───M──────────────

Circuito iSWAP (|10⟩ → i|01⟩):
0: ───X───iSwap───M('iswap')───
          │       │
1: ───────iSwap───M────────────

Resultado iSWAP:
iswap=0000000000, 1111111111


## Exemplo 5 — Algoritmo de Grover com Cirq

O **algoritmo de Grover** busca um item em N elementos em O(√N) operações,
vs. O(N) clássico — uma aceleração quadrática.

**Passos:**
1. **Superposição uniforme**: todos os estados com mesma probabilidade
2. **Oráculo**: marca o estado alvo (inverte sua fase)
3. **Difusão de Grover**: amplifica a amplitude do estado marcado
4. Repetir passos 2-3 cerca de **π/4 × √N** vezes

Aqui buscamos o estado `|11⟩` em um espaço de 4 estados (2 qubits).

In [5]:
import cirq
import numpy as np

q0, q1 = cirq.LineQubit.range(2)

def oraculo_11(q0, q1):
    """Marca o estado |11⟩ invertendo sua fase (aplica -1).
    CZ inverte a fase quando ambos os qubits são |1⟩."""
    return cirq.CZ(q0, q1)

def difusao_grover(q0, q1):
    """Operador de difusão de Grover: amplifica o estado marcado.
    Equivale a 2|s⟩⟨s| - I onde |s⟩ é a superposição uniforme."""
    return [
        cirq.H(q0), cirq.H(q1),
        cirq.X(q0), cirq.X(q1),
        cirq.CZ(q0, q1),
        cirq.X(q0), cirq.X(q1),
        cirq.H(q0), cirq.H(q1),
    ]

# Circuito de Grover
grover = cirq.Circuit([
    # 1. Superposição uniforme
    cirq.H(q0), cirq.H(q1),

    # 2. Oráculo: marca |11⟩
    oraculo_11(q0, q1),

    # 3. Difusão: amplifica |11⟩
    difusao_grover(q0, q1),

    # 4. Medição
    cirq.measure(q0, q1, key='busca'),
])

print('Circuito de Grover (busca |11⟩):')
print(grover)

sim = cirq.Simulator()
resultados = sim.run(grover, repetitions=1024)
contagens = resultados.histogram(key='busca')
print('\nContagens: {0=|00⟩, 1=|01⟩, 2=|10⟩, 3=|11⟩}')
print(contagens)
# 3 (= |11⟩) deve dominar com ~100% de probabilidade

Circuito de Grover (busca |11⟩):
0: ───H───@───H───X───@───X───H───M('busca')───
          │           │           │
1: ───H───@───H───X───@───X───H───M────────────

Contagens: {0=|00⟩, 1=|01⟩, 2=|10⟩, 3=|11⟩}
Counter({3: 1024})


## Exemplo 6 — Ruído e Canais Quânticos

O Cirq também suporta simulação de ruído realista usando
**canais quânticos** (operadores de Kraus).

### Tipos de ruído
| Canal | Efeito |
|-------|--------|
| `DepolarizingChannel(p)` | Ruído isotrópico geral |
| `BitFlipChannel(p)` | Flip de qubit com prob p |
| `PhaseFlipChannel(p)` | Flip de fase com prob p |
| `AmplitudeDampingChannel(γ)` | Decaimento de energia (T1) |

Com Cirq você pode inserir ruído **diretamente no circuito** como uma operação.

In [6]:
import cirq

q0, q1 = cirq.LineQubit.range(2)

# --- Circuito Bell com ruído inserido explicitamente ---
p_erro = 0.08  # 8% de probabilidade de erro

circuit_ruidoso = cirq.Circuit([
    # Porta H + canal de ruído depolarizante
    cirq.H(q0),
    cirq.depolarize(p=p_erro).on(q0),           # ruído em q0

    # CNOT + canal de bit flip
    cirq.CNOT(q0, q1),
    cirq.bit_flip(p=p_erro/2).on(q0),           # ruído no controle
    cirq.bit_flip(p=p_erro/2).on(q1),           # ruído no alvo

    cirq.measure(q0, q1, key='resultado'),
])

print('Circuito com canais de ruído explícitos:')
print(circuit_ruidoso)

# DensityMatrixSimulator: necessário para simular canais de ruído
# (o Simulator padrão assume evolução unitária pura)
sim_ruidoso = cirq.DensityMatrixSimulator()
resultado = sim_ruidoso.run(circuit_ruidoso, repetitions=2000)

contagens = resultado.histogram(key='resultado')
total = sum(contagens.values())
print('\nResultados com ruído (0=|00⟩, 1=|01⟩, 2=|10⟩, 3=|11⟩):')
for estado, count in sorted(contagens.items()):
    simbolo = ['|00⟩','|01⟩','|10⟩','|11⟩'][estado]
    print(f'  {simbolo}: {count:4d} ({100*count/total:.1f}%)')
print('\nEstados 01 e 10 = erros causados pelo ruído inserido')

Circuito com canais de ruído explícitos:
0: ───H───D(0.08)───@───BF(0.04)───M('resultado')───
                    │              │
1: ─────────────────X───BF(0.04)───M────────────────

Resultados com ruído (0=|00⟩, 1=|01⟩, 2=|10⟩, 3=|11⟩):
  |00⟩:  934 (46.7%)
  |01⟩:   83 (4.2%)
  |10⟩:   84 (4.2%)
  |11⟩:  899 (45.0%)

Estados 01 e 10 = erros causados pelo ruído inserido


## API Essencial do Cirq

| Objeto / Função | Papel |
|-----------------|-------|
| `cirq.LineQubit.range(n)` | Cria n qubits em linha |
| `cirq.GridQubit(row, col)` | Qubit em grade 2D |
| `cirq.Circuit([...])` | Constrói circuito |
| `cirq.Moment([...])` | Define camada explícita de portas |
| `cirq.H`, `cirq.CNOT`, `cirq.CZ` | Portas quânticas |
| `cirq.measure(q, key='k')` | Medição com nome |
| `sim.simulate(c)` | Retorna statevector exato |
| `sim.run(c, repetitions=N)` | N shots, retorna medições |
| `resultado.histogram(key='k')` | Contagem de resultados |